# Queue: K-writer ablation — DictaGen bigram set, K = 6, 7, 8, 9 of 10 writers, 1k / 10k / 50k x 5 writer draws (60 runs)

Same protocol as the 5-writer queue (exp39_5w): LM1, same training steps as the 10-writer runs, training seed 42, old HHD test. Order: all 1k runs (K = 6..9), then 10k, then 50k. Each run pushes its model + results to its own public HF repo; a run whose repo already holds `results.json` is skipped, so re-running resumes the queue. A failed run's executed notebook is uploaded to the public HF dataset `cyttic/a5000-queue-logs`. The local run folder is deleted after a successful push.

Start on the server: `./run.sh exp39_6to9w/queue_6to9w.ipynb` (inside tmux).


In [ ]:
import os, json, time, traceback, shutil
import papermill as pm
from huggingface_hub import HfApi
import torch
assert torch.cuda.is_available(), 'NO GPU visible (nvidia-smi?) - refusing to run the queue on CPU'   # 2026-11-08: VM lost the A5000, runs fell back to CPU
print('GPU:', torch.cuda.get_device_name(0), flush=True)
NBS = [
 "exp39_human10bi_hhdgeom_llmgen_6w_d1_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d2_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d3_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d4_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d5_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d1_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d2_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d3_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d4_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d5_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d1_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d2_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d3_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d4_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d5_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d1_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d2_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d3_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d4_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d5_1k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d1_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d2_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d3_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d4_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d5_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d1_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d2_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d3_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d4_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d5_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d1_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d2_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d3_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d4_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d5_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d1_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d2_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d3_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d4_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d5_10k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d1_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d2_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d3_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d4_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_6w_d5_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d1_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d2_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d3_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d4_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_7w_d5_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d1_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d2_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d3_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d4_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_8w_d5_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d1_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d2_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d3_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d4_50k_colab.ipynb",
 "exp39_human10bi_hhdgeom_llmgen_9w_d5_50k_colab.ipynb"
]
HERE = os.getcwd(); BASE = os.path.expanduser('~/work/queue_6to9w'); os.makedirs(BASE, exist_ok=True)
api = HfApi()
LOGS = 'cyttic/a5000-queue-logs'   # PUBLIC dataset: executed notebooks of failed runs (tracebacks)
api.create_repo(LOGS, repo_type='dataset', private=False, exist_ok=True)

def push_log(path, name):
    try: api.upload_file(path_or_fileobj=path, path_in_repo=f'{name}.ipynb', repo_id=LOGS, repo_type='dataset', commit_message=f'failed run {name}'); print('log ->', f'{LOGS}/{name}.ipynb', flush=True)
    except Exception as e: print('log upload failed', repr(e)[:200], flush=True)

def out_repo(nb):
    src = ''.join(''.join(c['source']) for c in json.load(open(os.path.join(HERE, nb)))['cells'])
    return src.split('OUT_REPO = "', 1)[1].split('"', 1)[0]

def done(repo):
    try: return 'results.json' in api.list_repo_files(repo)
    except Exception: return False

status = {}
for i, nb in enumerate(NBS, 1):
    repo = out_repo(nb); name = nb.replace('_colab.ipynb', '')
    if done(repo): print(f'[{i}/{len(NBS)}] {name}: already on HF, skipped', flush=True); status[name] = 'skipped (done)'; continue
    wd = os.path.join(BASE, name); os.makedirs(wd, exist_ok=True); t0 = time.time()
    print(f'[{i}/{len(NBS)}] {name} -> {repo}  started {time.strftime("%H:%M")}', flush=True)
    try:
        pm.execute_notebook(os.path.join(HERE, nb), os.path.join(wd, nb), cwd=wd, log_output=True, kernel_name='python3')
        status[name] = f'ok ({(time.time() - t0) / 3600:.2f} h)' if done(repo) else 'finished, but results.json not on HF'
        if done(repo): shutil.rmtree(wd, ignore_errors=True)   # pushed: free the disk (60 runs x ~0.35 GB of weights)
    except Exception as e:
        status[name] = 'FAILED: ' + repr(e)[:300]; traceback.print_exc(); push_log(os.path.join(wd, nb), 'queue_6to9w__' + name)
    print(f'[{i}/{len(NBS)}] {name}: {status[name]}', flush=True)
print('\n==== queue summary'); [print(f'{k:60s} {v}') for k, v in status.items()]

## Results on HF

In [ ]:
from huggingface_hub import hf_hub_download
for nb in NBS:
    repo = out_repo(nb)
    if not done(repo): print(f'{repo}: no results'); continue
    T = json.load(open(hf_hub_download(repo, 'results.json', force_download=True)))['final']['test_decoding']
    print(repo, {k.split(' (')[0]: round(100 * v['cer'], 2) for k, v in T.items()})